# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library with full Croissant-schema support by referencing all entities (record sets, fields, etc.) **by their `@id`**.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = json.loads(dataset.metadata.to_json())
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Let's review the available record sets, fields, and their IDs. For full reproducibility, we reference all entities by their `@id`.

In [ ]:
# Explore available record sets and fields by id
record_sets = list(dataset.record_sets)
if len(record_sets) == 0:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']} | name: {rs.get('name', 'N/A')}")
        # List fields within this record set
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            print(f"    Field: {field['@id']} | name: {field.get('name', 'N/A')}")

If no record sets print in the above cell, the dataset may store tabular or regression results as *one or more file-based record sets* and you can retrieve them via `dataset.records()` without specifying a record set. Otherwise, use the record set and field `@id`s listed above in your analyses.

## 3. Data Extraction
Let's attempt to extract data for available record sets based on their `@id`. If the dataset contains no explicit record sets, we will extract records as data entries.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Attempt to list all available record set @ids for extraction
record_sets = list(dataset.record_sets)
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
if record_set_ids:
    print('Extracting all available RecordSets by @id:')
    for record_set_id in record_set_ids:
        print(f"  {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if not records:
            print(f"    No records found for RecordSet {record_set_id}.")
        else:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
else:
    print("No explicit RecordSets found. Attempting to load records without specifying record_set...")
    records = list(dataset.records())
    if not records:
        print("No records could be extracted from the dataset.")
    else:
        df = pd.DataFrame(records)
        anon_recordset = 'main-unnamed-recordset'
        dataframes[anon_recordset] = df
        record_set_ids = [anon_recordset]

# Print columns from the first available dataframe
example_record_set_id = record_set_ids[0]
if example_record_set_id in dataframes:
    print(f"\nColumns in record set (@id): {example_record_set_id}")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Here we will:
- Filter records based on the value of a numeric field (e.g., likelihood, coefficient value, etc.).
- Normalize this field, then group by a categorical field if available (for example, variable name or knowledge type).

We'll reference all columns (fields) by their `@id` or exact titles.

In [ ]:
# Choose record set and fields by id or column name

# Use the first loaded record set for analysis
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]
print(f"Using record set: {record_set_id}")

# From display above, pick numeric and grouping fields
numeric_candidates = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()

if not numeric_candidates:
    print("No numeric fields found for EDA.")
    numeric_field = None
else:
    numeric_field = numeric_candidates[0]
    print(f"Numeric field selected: {numeric_field}")

if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping field selected: {group_field}")
else:
    group_field = None

if numeric_field:
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalize field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group by a group field if available
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped {numeric_field} mean by {group_field}:")
        display(grouped_df.head())
else:
    print("Skipping numeric analysis due to lack of numeric fields.")

## 5. Visualization
Let's visualize the normalized numeric field and, if available, group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[norm_col], kde=True)
    plt.title(f'Normalized Distribution of {numeric_field} (RecordSet {record_set_id})')
    plt.xlabel(norm_col)
    plt.ylabel('Count')
    plt.show()
    
    # If grouped summary available, plot group means
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field])
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load metadata and tabular records from a Croissant FAIR^2 dataset using `mlcroissant`.
- Examine record sets, fields, and extract data using their `@id`.
- Conduct quick exploratory analysis including filtering, normalizing, and grouping by fields.
- Visualize key variables and relationships.

This approach ensures reproducibility, allows scalable schema-based pipelines, and prepares your data for ML and statistical modeling inline with the FAIR data principles.

For more details on field and record set `@id`s, revisit the Data Overview section above.